In [ ]:
from ultralytics import YOLO
import numpy
import matplotlib

In [1]:
import os
import shutil
import random
from pathlib import Path
from collections import Counter

random.seed(42)

# ── Configure your paths here ──────────────────────────────────────────────
DATASETS = {
    "indoor":       r"D:\DW\Cse468 Project\path\to\indoor",
    "d_fire":      r"D:\DW\Cse468 Project\path\to\d_fire",
    "flame_bd9xu_smoll": r"D:\DW\Cse468 Project\path\to\flame_bd9xu_smoll",
    "iron_wolf_yvovn":        r"D:\DW\Cse468 Project\path\to\iron_wolf_yvovn",
}
OUTPUT_DIR = r"D:\DW\Cse468 Project\path\to\final_merged_3"

# ── Class remapping for each dataset ──────────────────────────────────────
# Fill these in AFTER running the audit above
# Format: {original_class_id: new_class_id}  (-1 = discard)
# Target: 0=fire, 1=smoke
REMAPS = {
    "indoor":           {0: 0, 1: 1},
    "d_fire":          {0: 1, 1: 0},
    "flame_bd9xu_smoll":      {0: 0},  # fire→fire, other→discard, smoke→smoke
    "iron_wolf_yvovn":  {0: 0, 1: -1, 2: -1, 3: 1},
}

# ── Collect all images from all datasets ──────────────────────────────────
all_samples = []  # list of (img_path, lbl_path, dataset_name)

for ds_name, ds_root in DATASETS.items():
    root = Path(ds_root)
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = root / split / 'images' 
        lbl_dir = root / split /'labels'
        if not img_dir.exists():
            continue

        for img_path in img_dir.iterdir():
            if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                continue
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if lbl_path.exists():
                all_samples.append((img_path, lbl_path, ds_name))

print(f"Total samples collected: {len(all_samples)}")

# ── Shuffle and split 70/20/10 ────────────────────────────────────────────
random.shuffle(all_samples)
n = len(all_samples)
splits = {
    'train': all_samples[:int(n * 0.70)],
    'val':   all_samples[int(n * 0.70):int(n * 0.90)],
    'test':  all_samples[int(n * 0.90):]
}

# ── Copy files to output with remapping ───────────────────────────────────
class_counts = Counter()

for split_name, samples in splits.items():
    img_out = Path(OUTPUT_DIR) / split_name/ 'images'
    lbl_out = Path(OUTPUT_DIR) / split_name/ 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_path, lbl_path, ds_name in samples:
        remap = REMAPS[ds_name]

        # Remap labels
        new_lines = []
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            orig_cls = int(parts[0])
            new_cls = remap.get(orig_cls, -1)
            if new_cls == -1:
                continue
            new_lines.append(f"{new_cls} {' '.join(parts[1:])}")
            class_counts[new_cls] += 1

        if not new_lines:
            continue  # skip images with no valid annotations

        # Prefix filename with dataset name to avoid collisions
        new_stem = f"{ds_name}_{img_path.stem}"
        shutil.copy(img_path, img_out / (new_stem + img_path.suffix))
        (lbl_out / (new_stem + '.txt')).write_text('\n'.join(new_lines))

print(f"\nMerge complete.")
print(f"Fire annotations:  {class_counts[0]}")
print(f"Smoke annotations: {class_counts[1]}")
print(f"Fire:Smoke ratio:  {class_counts[0]/max(class_counts[1],1):.2f}:1")
print(f"\nSplit sizes:")
for s, samples in splits.items():
    print(f"  {s}: {len(samples)} images")

Total samples collected: 45252

Merge complete.
Fire annotations:  42138
Smoke annotations: 20757
Fire:Smoke ratio:  2.03:1

Split sizes:
  train: 31676 images
  val: 9050 images
  test: 4526 images


In [ ]:
import os
import random
import shutil

# ==== CONFIG ====
BASE_DIR = "flame_bd9xu"  # root folder
OUTPUT_DIR = "flame_bd9xu_smoll"

SAMPLE_TOTAL = 2000
SEED = 42

SPLITS = {
    "train": 0.7,
    "val": 0.2,
    "test": 0.1
}

random.seed(SEED)

def process_split(split, ratio):
    img_dir = os.path.join(BASE_DIR, split, "images")
    lbl_dir = os.path.join(BASE_DIR, split, "labels")

    out_img_dir = os.path.join(OUTPUT_DIR, split, "images")
    out_lbl_dir = os.path.join(OUTPUT_DIR, split, "labels")

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    images = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

    sample_size = int(SAMPLE_TOTAL * ratio)
    sample_size = min(sample_size, len(images))

    sampled = random.sample(images, sample_size)

    missing_labels = 0

    for img_name in sampled:
        # Copy image
        shutil.copy(os.path.join(img_dir, img_name),
                    os.path.join(out_img_dir, img_name))

        # Copy label
        label_name = os.path.splitext(img_name)[0] + ".txt"
        src_label = os.path.join(lbl_dir, label_name)

        if os.path.exists(src_label):
            shutil.copy(src_label,
                        os.path.join(out_lbl_dir, label_name))
        else:
            missing_labels += 1

    print(f"{split}: {len(sampled)} sampled, {missing_labels} missing labels")


# ==== RUN ====
for split, ratio in SPLITS.items():
    process_split(split, ratio)

print("Done.")

train: 1050 sampled, 13 missing labels
val: 300 sampled, 0 missing labels
test: 101 sampled, 0 missing labels
Done.
